# Deliverable 3: Supervised Modeling — Two-Stage Delay Pipeline

**Research Question:** At the moment of departure, can we (1) identify which flights will arrive 15+ minutes late, and (2) for those flagged flights, how late will they arrive?

**Pipeline:**
- **Stage 1 — Random Forest Classifier:** Trained on all flights. Features include pre-departure schedule info + `dep_delay` (known at pushback). Predicts binary delayed/not-delayed.
- **Stage 2 — XGBoost Regressor:** Trained on delayed flights only. Predicts exact arrival delay in minutes.

**Leakage guard:** `dep_delay` is included (known at gate before wheels-off). All post-departure columns (taxi_out, wheels_off/on, air_time, arr_time) and all delay attribution columns (carrier_delay, weather_delay, nas_delay, security_delay, late_aircraft_delay) are excluded.


In [ ]:
import pandas as pd
import numpy as np
import warnings, json
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, KFold,
                                     cross_validate, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

print("Imports OK")


## 1. Load & Clean Data

In [ ]:
DATA_PATH = r'C:\Users\erica\stat486proj\code\non_cancelled_flights.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape}")

df = df.dropna(subset=['arr_delay', 'dep_delay'])
print(f"After dropping missing targets: {len(df)} rows")
print(f"\narr_delay range: {df['arr_delay'].min():.0f} to {df['arr_delay'].max():.0f} min")
print(f"dep_delay range: {df['dep_delay'].min():.0f} to {df['dep_delay'].max():.0f} min")


## 2. Feature Engineering

In [ ]:
# Extract hour from HHMM integers
df['dep_hour']   = df['crs_dep_time'] // 100
df['arr_hour']   = df['crs_arr_time'] // 100
df['is_weekend'] = df['day_of_week'].isin([6, 7]).astype(int)

# ── Feature set (available at gate/pushback) ──────────────────────────────────
FEATURES = [
    'month', 'day_of_month', 'day_of_week', 'dep_hour', 'arr_hour',
    'is_weekend', 'crs_elapsed_time', 'distance',
    'dep_delay',                         # known at pushback
    'op_unique_carrier', 'origin', 'dest'
]

CAT_COLS = ['op_unique_carrier', 'origin', 'dest']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

# ── Stage 1: all flights ──────────────────────────────────────────────────────
X_all = df[FEATURES].copy()
y_clf = (df['arr_delay'] >= 15).astype(int)

# ── Stage 2: delayed flights only ─────────────────────────────────────────────
mask_delayed = df['arr_delay'] >= 15
X_del = df.loc[mask_delayed, FEATURES].copy()
p99   = df.loc[mask_delayed, 'arr_delay'].quantile(0.99)
y_reg = df.loc[mask_delayed, 'arr_delay'].clip(upper=p99)

print(f"Stage 1 — all flights:   {len(X_all):,} rows, {y_clf.mean():.1%} delayed")
print(f"Stage 2 — delayed only:  {len(X_del):,} rows")
print(f"Regression target clipped at {p99:.0f} min (99th pct)")
print(f"\ndep_delay → arr_delay correlation (delayed only): "
      f"{df.loc[mask_delayed,'dep_delay'].corr(df.loc[mask_delayed,'arr_delay']):.3f}")


## 3. Shared Preprocessor (inside Pipeline)

In [ ]:
def make_preprocessor():
    """Returns a fresh ColumnTransformer — one per pipeline to avoid state sharing."""
    return ColumnTransformer([
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1), CAT_COLS)
    ], remainder='drop')

print("Preprocessor factory defined.")
print(f"Numeric features:    {NUM_COLS}")
print(f"Categorical features: {CAT_COLS}")


## 4. Stage 1 — Random Forest Classifier

Trained on all 9,836 flights. Tuned via `GridSearchCV` with `StratifiedKFold(k=5)` to handle the 21.5% positive class rate. Scoring optimized on ROC-AUC; `class_weight='balanced'` used to avoid majority-class bias.


In [ ]:
rf_pipe = Pipeline([
    ('pre', make_preprocessor()),
    ('clf', RandomForestClassifier(random_state=42,
                                   class_weight='balanced', n_jobs=-1))
])

param_grid_rf = {
    'clf__n_estimators':      [100, 200],
    'clf__max_depth':         [5, 10, None],
    'clf__min_samples_split': [2, 10],
}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs_rf = GridSearchCV(rf_pipe, param_grid_rf, cv=cv_strat,
                     scoring='roc_auc', n_jobs=-1, refit=True, verbose=1)
gs_rf.fit(X_all, y_clf)
best_rf = gs_rf.best_estimator_

print(f"Best RF params: {gs_rf.best_params_}")


In [ ]:
rf_cv = cross_validate(best_rf, X_all, y_clf, cv=cv_strat,
                       scoring=['accuracy', 'f1', 'roc_auc'],
                       return_train_score=True)

print(f"CV Accuracy  : {rf_cv['test_accuracy'].mean():.3f} ± {rf_cv['test_accuracy'].std():.3f}")
print(f"CV F1        : {rf_cv['test_f1'].mean():.3f} ± {rf_cv['test_f1'].std():.3f}")
print(f"CV ROC-AUC   : {rf_cv['test_roc_auc'].mean():.3f} ± {rf_cv['test_roc_auc'].std():.3f}")
print(f"Train ROC-AUC: {rf_cv['train_roc_auc'].mean():.3f}  (overfitting gap: "
      f"{rf_cv['train_roc_auc'].mean() - rf_cv['test_roc_auc'].mean():.3f})")


## 5. Stage 2 — XGBoost Regressor (delayed flights only)

Trained exclusively on the 2,119 flights with arr_delay ≥ 15 min. Tuned via `GridSearchCV` with `KFold(k=5)`, scoring on neg RMSE. The 99th-percentile cap (418 min) reduces influence of extreme multi-hour outliers.


In [ ]:
xgb_pipe = Pipeline([
    ('pre', make_preprocessor()),
    ('reg', xgb.XGBRegressor(random_state=42, verbosity=0, n_jobs=-1))
])

param_grid_xgb = {
    'reg__n_estimators':      [100, 200],
    'reg__max_depth':         [4, 6],
    'reg__learning_rate':     [0.05, 0.1],
    'reg__subsample':         [0.8, 1.0],
    'reg__colsample_bytree':  [0.8, 1.0],
}

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)

gs_xgb = GridSearchCV(xgb_pipe, param_grid_xgb, cv=cv_kfold,
                      scoring='neg_root_mean_squared_error',
                      n_jobs=-1, refit=True, verbose=1)
gs_xgb.fit(X_del, y_reg)
best_xgb = gs_xgb.best_estimator_

print(f"Best XGB params: {gs_xgb.best_params_}")


In [ ]:
xgb_cv = cross_validate(best_xgb, X_del, y_reg, cv=cv_kfold,
                        scoring=['neg_root_mean_squared_error',
                                 'neg_mean_absolute_error', 'r2'],
                        return_train_score=True)

rmse = -xgb_cv['test_neg_root_mean_squared_error'].mean()
mae  = -xgb_cv['test_neg_mean_absolute_error'].mean()
r2   =  xgb_cv['test_r2'].mean()

print(f"CV RMSE  : {rmse:.2f} ± {-xgb_cv['test_neg_root_mean_squared_error'].std():.2f} min")
print(f"CV MAE   : {mae:.2f} min")
print(f"CV R²    : {r2:.3f} ± {xgb_cv['test_r2'].std():.3f}")
print(f"Train R² : {xgb_cv['train_r2'].mean():.3f}  (overfitting gap: "
      f"{xgb_cv['train_r2'].mean() - r2:.3f})")


## 6. Explainability & Interpretability

- **Stage 1:** Gini feature importances from the Random Forest
- **Stage 2:** SHAP values from XGBoost (TreeExplainer, 500-flight sample of delayed subset)


In [ ]:
# Refit on full data for importance extraction
best_rf.fit(X_all, y_clf)
best_xgb.fit(X_del, y_reg)

feature_names = NUM_COLS + CAT_COLS

# SHAP
X_shap   = X_del.sample(min(500, len(X_del)), random_state=42)
X_shap_t = best_xgb['pre'].transform(X_shap)
explainer = shap.TreeExplainer(best_xgb['reg'])
shap_vals = explainer.shap_values(X_shap_t)
mean_shap = np.abs(shap_vals).mean(axis=0)
shap_order = np.argsort(mean_shap)[::-1]

# RF importances
rf_imp   = best_rf['clf'].feature_importances_
rf_order = np.argsort(rf_imp)[::-1]

n = len(feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f1117')
BG = '#1a1d2e'

axes[0].barh(
    [feature_names[i] for i in rf_order[::-1]],
    rf_imp[rf_order[::-1]],
    color=plt.cm.Greens(np.linspace(0.35, 0.9, n))
)
axes[0].set_facecolor(BG)
axes[0].tick_params(colors='white', labelsize=9)
axes[0].set_xlabel('Gini Importance', color='white', fontsize=10)
axes[0].set_title('Stage 1 — Random Forest\nFeature Importance (Classification)',
                  color='white', fontweight='bold')
for sp in axes[0].spines.values(): sp.set_edgecolor('#444')

axes[1].barh(
    [feature_names[i] for i in shap_order[::-1]],
    mean_shap[shap_order[::-1]],
    color=plt.cm.Blues(np.linspace(0.35, 0.9, n))
)
axes[1].set_facecolor(BG)
axes[1].tick_params(colors='white', labelsize=9)
axes[1].set_xlabel('Mean |SHAP value| (minutes)', color='white', fontsize=10)
axes[1].set_title('Stage 2 — XGBoost SHAP\nFeature Importance (Regression)',
                  color='white', fontweight='bold')
for sp in axes[1].spines.values(): sp.set_edgecolor('#444')

plt.tight_layout(pad=2)
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f"Top RF feature  (Stage 1): {feature_names[rf_order[0]]}")
print(f"Top SHAP feature (Stage 2): {feature_names[shap_order[0]]}")


## 7. Pipeline Summary

| Stage | Model | Metric | Value |
|---|---|---|---|
| Stage 1 | Random Forest | CV Accuracy | 0.920 |
| Stage 1 | Random Forest | CV F1 | 0.807 |
| Stage 1 | Random Forest | CV ROC-AUC | 0.921 |
| Stage 2 | XGBoost | CV RMSE | 17.9 min |
| Stage 2 | XGBoost | CV MAE | 12.3 min |
| Stage 2 | XGBoost | CV R² | 0.935 |

**Key insight:** `dep_delay` is the dominant feature in both stages — it acts as a proxy for all upstream disruptions (weather, ATC, mechanical) that caused the departure to slip. Route characteristics (`distance`, `crs_elapsed_time`, `dep_hour`) modulate how much of that delay is recovered or amplified in flight.

**Comparison to baseline (pure schedule features, no dep_delay):**
- Stage 1 ROC-AUC: 0.643 → **0.921** (+0.278)
- Stage 2 R²: 0.045 → **0.935** (+0.890)
